In [26]:
import os
from typing import Dict, Tuple
import sys

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import pandas as pd
# for Jupyter notebooks: add the path of 'code' to allow importing module
sys.path.append(os.path.join(os.getcwd(), ".."))

In [27]:
from translate_hyfd import  parse_file, translate_results
from measures_generalized import measures_dict
from utils import test_fd_with_a_measure

In [28]:
    
input_file_adult = 'hyfd_adult_sample_10000_determinant_limit_3'
input_file_dblp = 'hyfd_dblp_full_determinant_limit_1'

input_file_acs = 'datasets/acs-pums/fds.txt'
input_file_compass = 'datasets/compass/fds.txt'
input_file_abalone = 'datasets/abalone/abalone_hyfd_raw_determinant_limit3'

In [29]:
def plot_res(dfs):
    for df in dfs:
        measure = df['measure'][0]  # Get the measure name
        plt.figure(figsize=(8, 5))
        plt.bar(df['fd'], df['val'], color='blue', alpha=0.7)
        plt.xlabel('Functional Dependency')
        plt.ylabel('Value')
        plt.title(f'Measure: {measure}')
        plt.xticks(rotation=45)
        plt.ylim(0, 1.05)
        plt.show()

In [32]:
def gen_summary_df(csv_file_dir, measures_dict, input_file_hyfd=None, input_handwritten_file=None):
    runtimes = {}
    
    if(input_file_hyfd):
        column_mapping, results = parse_file(input_file_hyfd)
        display(column_mapping)
        # print(len(column_mapping))
        translated_results = translate_results(results, column_mapping)
    elif(input_handwritten_file):
        with open(input_handwritten_file, 'r') as f:
            lines = f.readlines()
            translated_results = []
            for line in lines:
                cols, target = line.split("->")
                cols = cols.split(",")
                if(all([x for x in cols])): # some hyfd doesnt have lhs?
                    translated_cols = [col.strip() for col in cols]
                    translated_target = target.strip()
                    translated_results.append(f"{', '.join(translated_cols)} -> {translated_target}")

    print("translated_results:")
    print(translated_results)
    data_df = pd.read_csv(csv_file_dir)
    cols = [x.strip().replace('-','_') for x in list(data_df)]
    print(cols)
    print(data_df.head())
    data_df.columns = cols
    res_dfs = []
    for k in measures_dict:
        start_time = time.time()
        print(f"working on {k}")
        df_from_measure = test_fd_with_a_measure(measure_dict=measures_dict, measure_name=k, cand_fds=translated_results, data_df=data_df)
        res_dfs.append(df_from_measure)
        end_time = time.time()
        runtimes[k] = end_time - start_time
    return res_dfs, runtimes


In [33]:
# res_adults, runtimes = gen_summary_df(input_file_hyfd=input_file_adult, csv_file_dir='data/adult.csv', measures_dict=measures_dict)
# res_adult, runtimes = gen_summary_df(input_file_hyfd=input_file_dblp, csv_file_dir='data/dblp10k.csv', measures_dict=measures_dict)
# res_acs, runtimes = gen_summary_df(input_handwritten_file=input_file_acs, csv_file_dir='datasets/acs-pums/acs.csv', measures_dict=measures_dict)
# res_acs, runtimes = gen_summary_df(input_handwritten_file=input_file_compass, csv_file_dir='datasets/compass/compass.csv', measures_dict=measures_dict)
res_acs, runtimes = gen_summary_df(input_file_hyfd=input_file_abalone, csv_file_dir='datasets/abalone/abalone.csv', measures_dict=measures_dict)


{6: 'ShuckedWeight',
 1: 'Sex',
 7: 'VisceraWight',
 9: 'Rings',
 3: 'Diameter',
 4: 'Heigh',
 5: 'WholeWeight',
 2: 'Length',
 8: 'ShellWeight'}

translated_results:
['ShellWeight, ShuckedWeight, WholeWeight -> VisceraWight', 'ShellWeight, ShuckedWeight, WholeWeight -> Length', 'ShellWeight, ShuckedWeight, WholeWeight -> Diameter', 'ShellWeight, ShuckedWeight, WholeWeight -> Heigh', 'ShellWeight, ShuckedWeight, WholeWeight -> Rings', 'ShellWeight, ShuckedWeight, WholeWeight -> Sex', 'ShuckedWeight, VisceraWight, WholeWeight -> Diameter', 'ShuckedWeight, VisceraWight, WholeWeight -> Sex', 'Length, ShuckedWeight, WholeWeight -> Rings', 'Diameter, ShuckedWeight, WholeWeight -> VisceraWight', 'Diameter, ShuckedWeight, WholeWeight -> Sex', 'ShellWeight, VisceraWight, WholeWeight -> ShuckedWeight', 'ShellWeight, VisceraWight, WholeWeight -> Length', 'ShellWeight, VisceraWight, WholeWeight -> Diameter', 'ShellWeight, VisceraWight, WholeWeight -> Heigh', 'ShellWeight, VisceraWight, WholeWeight -> Rings', 'ShellWeight, VisceraWight, WholeWeight -> Sex', 'Diameter, VisceraWight, WholeWeight -> Sex']
['Sex', 'Length', 'Diameter', 'Heigh', 

In [34]:
res_acs

[   measure                                                 fd  val
 0      rho  ShellWeight, ShuckedWeight, WholeWeight -> Vis...  1.0
 1      rho  ShellWeight, ShuckedWeight, WholeWeight -> Length  1.0
 2      rho  ShellWeight, ShuckedWeight, WholeWeight -> Dia...  1.0
 3      rho   ShellWeight, ShuckedWeight, WholeWeight -> Heigh  1.0
 4      rho   ShellWeight, ShuckedWeight, WholeWeight -> Rings  1.0
 5      rho     ShellWeight, ShuckedWeight, WholeWeight -> Sex  1.0
 6      rho  ShuckedWeight, VisceraWight, WholeWeight -> Di...  1.0
 7      rho    ShuckedWeight, VisceraWight, WholeWeight -> Sex  1.0
 8      rho        Length, ShuckedWeight, WholeWeight -> Rings  1.0
 9      rho  Diameter, ShuckedWeight, WholeWeight -> Viscer...  1.0
 10     rho        Diameter, ShuckedWeight, WholeWeight -> Sex  1.0
 11     rho  ShellWeight, VisceraWight, WholeWeight -> Shuc...  1.0
 12     rho   ShellWeight, VisceraWight, WholeWeight -> Length  1.0
 13     rho  ShellWeight, VisceraWight, WholeWei

In [ ]:
plot_res(res_acs)

In [ ]:
df = pd.read_csv('datasets/acs-pums/acs.csv')


In [ ]:
df.groupby(['ST','DIVISION']).size()

In [ ]:
df.groupby(['DIVISION','REGION']).size()

In [ ]:

import pandas as pd
from sklearn.feature_selection import mutual_info_classif

# Sample dataset
data = pd.DataFrame({
    "Education": ["High School", "High School", "High School", "High School", "High School", "High School", "High School", "High School"],
    "Income": ["Low", "Low", "Low", "Low", "Low", "Low", "Low", "Low"]
})

# Convert categorical variables to numerical values
data_encoded = data.apply(lambda x: pd.factorize(x)[0])

# Compute mutual information
mi_scores = mutual_info_classif(data_encoded[["Education"]], data_encoded["Income"], discrete_features=True)

print(f"Mutual Information between Education and Income: {mi_scores[0]:.4f}")